# Confidence lab · exploring MiniWoB++

Runs in the lab's own environment (`labs/.venv`, kernel **lab**): `./labs/lab.sh` sets it up.

[MiniWoB++](https://github.com/Farama-Foundation/miniwob-plusplus): ~130 small web pages ("tasks"), each with a short instruction like *Click on the "submit" button.*

- **task page** = an HTML page + JS. The instruction and the page content are **generated at random** on each load, so a page alone isn't a test case.
- **demo** = one person doing one random episode of a task ([miniwob-plusplus-demos](https://github.com/stanfordnlp/miniwob-plusplus-demos)): the `utterance` (instruction), every mouse / key event, and the **whole DOM with boxes** at each moment.
- **gold element** of a click = the deepest DOM node under the click point. Nothing to guess.

Everything goes to `data/MiniWoB++/`.

## Download

In [1]:
#import subprocess
from pathlib import Path
#
DATA = Path("..") / "data" / "MiniWoB++"
#REPOS = {
#    "miniwob-plusplus": "https://github.com/Farama-Foundation/miniwob-plusplus",  # the task pages
#    "demos": "https://github.com/stanfordnlp/miniwob-plusplus-demos",              # human demos (~95 MB)
#}
#
#DATA.mkdir(parents=True, exist_ok=True)
#for name, url in REPOS.items():
#    dest = DATA / name
#    if dest.exists():
#        print(f"✔ {name} already there")
#        continue
#    print(f"⚡ cloning {name} …")
#    subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True)
#    print(f"✔ {name}")

## The task pages

In [2]:
import pandas as pd

TASK_DIR = DATA / "miniwob-plusplus" / "miniwob" / "html" / "miniwob"

pages = pd.DataFrame(
    [{"task": p.stem, "kb": round(p.stat().st_size / 1024, 1)} for p in sorted(TASK_DIR.glob("*.html"))]
)
print(len(pages), "task pages")
pages

130 task pages


,task,kb
0,ascending-numbers,3.3
1,bisect-angle,5.3
2,book-flight-nodelay,12.5
3,book-flight,12.5
4,button-delay,2.4
...,...,...
125,use-colorwheel,1.7
126,use-slider-2,2.8
127,use-slider,2.1
128,use-spinner,1.7


One page's source. Look for `genProblem`: it builds the page content and the instruction at random, each episode.

In [3]:
print((TASK_DIR / "click-button.html").read_text())

<!DOCTYPE html>
<html>
<head>
<title>Click Button Task</title>
<!-- stylesheets -->
<link rel="stylesheet" type="text/css" href="../core/core.css">
<!-- JS -->
<script src="../core/core.js"></script>
<script src="../core/d3.v3.min.js"></script>
<script src="../common/ui_utils.js"></script>
<script>
var buttons = ['submit', 'okay', 'ok', 'yes', 'no', 'cancel', 'next', 'previous'];

var genProblem = function() {
  // generate the task
  var div = d3.select('#area');
  div.html(''); // clear previous problem, if any

  var num = 6;
  var elements = [];
  for(var i=0;i<num;i++) {
    var u = core.randf(0,1);
    var must_make_button = (i === num-1) && elements.length === 0;

    if(u < 0.33 && !must_make_button) {
      var txt = ''; for(var q=0;q<3;q++) { txt += ' ' + core.sample(ui_utils.lorem_words); }
      div.append('div').html(txt);
    } else if(u < 0.66 && !must_make_button) {

      if(core.randf(0,1) < 0.5) {
        var txt = ''; for(var q=0;q<3;q++) { txt += ' ' + core.sample(

## The goal of each task

With MiniWoB's own environment (`miniwob` package): for each task, click every element once; the ones MiniWoB rewards are the goal.

In [4]:
import json
from pathlib import Path

import gymnasium
import miniwob
import pandas as pd
from miniwob.action import ActionTypes

gymnasium.register_envs(miniwob)

DATA = Path("..") / "data" / "MiniWoB++"

TASKS = sorted(name for name in gymnasium.registry if name.startswith("miniwob/"))
print(len(TASKS), "tasks")


def find_goal(name: str) -> dict:
    env = gymnasium.make(name)
    try:
        obs, _ = env.reset(seed=0)
        instruction = obs["utterance"]
        html = env.unwrapped.instance.driver.page_source
        elements = [e for e in obs["dom_elements"] if e["ref"] > 0]   # real elements (text pieces have ref < 0)

        goal = []
        for element in elements:
            env.reset(seed=0)                                          # same page again
            action = env.unwrapped.create_action(ActionTypes.CLICK_ELEMENT, ref=element["ref"])
            _, reward, _, _, info = env.step(action)
            if info.get("raw_reward", reward) > 0:
                goal.append({k: element[k] for k in ("ref", "tag", "text", "id", "classes")})

        return {"task": name, "instruction": instruction, "html": html,
                "elements": len(elements), "goal": goal}
    finally:
        env.close()

128 tasks


### Filter: only the tasks Thunder-X can do

Read each task's instruction, parse it with **Thunder-X's own `nl_parser`**, keep the task if:
- the parser recognises the action (click, hover, type, select, check…)
- it's **one step**: a single verb ("Enter X and press Submit" has two → dropped)

In [5]:
import re
import sys

sys.path.insert(0, str(Path("..").resolve()))          # Thunder-X's code
from thunderx.nl_parser import helpers as h
from thunderx.nl_parser.services import parse

ANY_VERB = re.compile(r"\b(?:" + "|".join(pattern for _, pattern in h.VERBS) + r")\b")


def instruction_of(name: str) -> str:
    env = gymnasium.make(name)
    try:
        obs, _ = env.reset(seed=0)
        return obs["utterance"]
    finally:
        env.close()


rows = []
for name in TASKS:
    try:
        instruction = instruction_of(name)
    except Exception as error:
        rows.append({"task": name, "instruction": "", "action": "", "verbs": 0,
                     "keep": False, "why": f"error: {error!r}"[:150]})
        continue
    action = parse(instruction).action
    verbs = len(ANY_VERB.findall(h.fold(instruction)))
    why = "action not recognised" if action is None else "several steps" if verbs > 1 else ""
    rows.append({"task": name, "instruction": instruction, "action": action or "", "verbs": verbs,
                 "keep": not why, "why": why})
filtered = pd.DataFrame(rows)

KEPT = filtered.loc[filtered["keep"], "task"].tolist()
print(f"{len(KEPT)} / {len(filtered)} tasks kept")
print(filtered.groupby(["keep", "why"]).size())
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 200)
filtered.sort_values(["keep", "why", "task"], ascending=[False, True, True])

127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /flight/AA/wrapper.html HTTP/1.1" 200 -
127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /core/core.css HTTP/1.1" 200 -
127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /flight/flight-common/wrapper.css HTTP/1.1" 200 -
127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /flight/flight-common/wrapper.js HTTP/1.1" 200 -
127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /core/core.js HTTP/1.1" 200 -
127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /flight/AA/dataset-AA.js HTTP/1.1" 200 -
127.0.0.1 - - [01/Nov/2026 21:11:58] code 404, message File not found
127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /flight/AA/index.html HTTP/1.1" 200 -
127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /flight/AA/content/common/css/v3/jquery-ui-1.10-aa.css HTTP/1.1" 200 -
127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /flight/AA/content/common/css/v3/core.css HTTP/1.1" 200 -
127.0.0.1 - - [01/Nov/2026 21:11:58] "GET /flight/AA/content/common/css/v

20 / 128 tasks kept
keep   why                  
False  action not recognised    74
       several steps            34
True                            20
dtype: int64


,task,instruction,action,verbs,keep,why
0,miniwob/ascending-numbers-v1,Click on the numbers in ascending order.,click,1,True,
12,miniwob/click-button-v1,"Click on the ""okay"" button.",click,1,True,
21,miniwob/click-color-v1,Click on the white colored box.,click,1,True,
22,miniwob/click-dialog-2-v1,"Click the button in the dialog box labeled ""Cancel"".",click,1,True,
24,miniwob/click-link-v1,"Click on the link ""Eget"".",click,1,True,
26,miniwob/click-menu-v1,Select Martelle,select,1,True,
32,miniwob/click-shape-v1,Click on a aqua item,click,1,True,
37,miniwob/click-tab-v1,Click on Tab #2.,click,1,True,
38,miniwob/click-test-2-v1,Click button ONE.,click,1,True,
39,miniwob/click-test-transfer-v1,Click button ONE.,click,1,True,


In [6]:
results = []
for name in KEPT:
    try:
        result = find_goal(name)
    except Exception as error:
        result = {"task": name, "error": repr(error)[:300]}
    results.append(result)
    print(f"{name:40} {len(result.get('goal', []))} goal element(s)")

miniwob/ascending-numbers-v1             0 goal element(s)
miniwob/click-button-v1                  2 goal element(s)
miniwob/click-color-v1                   1 goal element(s)
miniwob/click-dialog-2-v1                1 goal element(s)
miniwob/click-link-v1                    1 goal element(s)
miniwob/click-menu-v1                    2 goal element(s)
miniwob/click-shape-v1                   1 goal element(s)
miniwob/click-tab-v1                     1 goal element(s)
miniwob/click-test-2-v1                  1 goal element(s)
miniwob/click-test-transfer-v1           1 goal element(s)
miniwob/click-test-v1                    1 goal element(s)
miniwob/click-widget-v1                  6 goal element(s)
miniwob/count-sides-v1                   1 goal element(s)
miniwob/grid-coordinate-v1               1 goal element(s)
miniwob/identify-shape-v1                1 goal element(s)
miniwob/read-table-2-v1                  0 goal element(s)
miniwob/sign-agreement-v1                1 goal element(

In [7]:
tasks = pd.DataFrame(results)
tasks["goal_text"] = tasks["goal"].map(
    lambda g: " | ".join(f"<{e['tag']}> {e['text']!r}" for e in g) if isinstance(g, list) else ""
)

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 200)
print((tasks["goal_text"] != "").sum(), "/", len(tasks), "tasks with a goal")
tasks[["task", "instruction", "goal_text", "elements"]]

16 / 20 tasks with a goal


,task,instruction,goal_text,elements
0,miniwob/ascending-numbers-v1,Click on the numbers in ascending order.,,9
1,miniwob/click-button-v1,"Click on the ""okay"" button.",<button> 'okay' | <button> 'okay',9
2,miniwob/click-color-v1,Click on the white colored box.,<div> '',7
3,miniwob/click-dialog-2-v1,"Click the button in the dialog box labeled ""Cancel"".",<button> 'Cancel',22
4,miniwob/click-link-v1,"Click on the link ""Eget"".",<span> 'Eget',8
5,miniwob/click-menu-v1,Select Martelle,<li> '' | <div> 'Martelle',13
6,miniwob/click-shape-v1,Click on a aqua item,<text> 'O',7
7,miniwob/click-tab-v1,Click on Tab #2.,<a> 'Tab #2',12
8,miniwob/click-test-2-v1,Click button ONE.,<button> 'ONE',5
9,miniwob/click-test-transfer-v1,Click button ONE.,<button> 'ONE',5


In [8]:
# the HTML of one task (change the name to see another)
TASK = "miniwob/click-button-v1"
html = tasks.loc[tasks["task"] == TASK, "html"].iloc[0]
print(len(html), "characters")
print(html)

3178 characters
<html><head>
<title>Click Button Task</title>
<!-- stylesheets -->
<link rel="stylesheet" type="text/css" href="../core/core.css">
<!-- JS -->
<script src="../core/core.js"></script>
<script src="../core/d3.v3.min.js"></script>
<script src="../common/ui_utils.js"></script>
<script>
var buttons = ['submit', 'okay', 'ok', 'yes', 'no', 'cancel', 'next', 'previous'];

var genProblem = function() {
  // generate the task
  var div = d3.select('#area');
  div.html(''); // clear previous problem, if any

  var num = 6;
  var elements = [];
  for(var i=0;i<num;i++) {
    var u = core.randf(0,1);
    var must_make_button = (i === num-1) && elements.length === 0;

    if(u < 0.33 && !must_make_button) {
      var txt = ''; for(var q=0;q<3;q++) { txt += ' ' + core.sample(ui_utils.lorem_words); }
      div.append('div').html(txt);
    } else if(u < 0.66 && !must_make_button) {

      if(core.randf(0,1) < 0.5) {
        var txt = ''; for(var q=0;q<3;q++) { txt += ' ' + core.sample(u

In [ ]:
# Save the dataset: one table, one row per task → data/MiniWoB++/cases.parquet
#
# dataset = tasks.merge(filtered[["task", "action"]], on="task")[
#     ["task", "instruction", "action", "goal", "elements", "html"]
# ]
# dataset.to_parquet(DATA / "cases.parquet", index=False)
# print(len(dataset), "cases saved to", DATA / "cases.parquet")

In [ ]:
# i am going in cercles ; but 
# i know that 
# having different templates 
# being random at each call 
# with goal xpath and all
# will be a great database
# for a benchmark for this tool , ci pipelines , and live confidence betwwen steps

In [ ]:
"""
Thunder x: given a URL the tool can open and a natural-language instruction, it returns an actionable XPath for the element the instruction refers to.

to Benchmark I dataset where each row contains:

url: a page the tool can load
instruction: the natural-language command 
expected_xpath: a reference XPath that uniquely matches the target element and has been verified as actionable
"""

# Benchmark builder: (url, instruction, expected_xpath)

**Target first, instruction second**: the right answer is known by construction.

1. **read** the page with Thunder-X's `page_extractor`
2. **pick** target elements and an action for each
3. **XPath**: try simple rules, keep the first that matches exactly the target
4. **instruction**: written from a template ("Click on 'Login'", "Type 'Jane' into the 'Name' field"…)
5. **save** to a SQLite file

In [1]:
import os
from datetime import datetime
from pathlib import Path

URLS = [
    "https://www.selenium.dev/selenium/web/web-form.html",
    "https://testpages.eviltester.com/styled/basic-html-form-test.html",
    "https://the-internet.herokuapp.com/login",
    "https://the-internet.herokuapp.com/checkboxes",
    "https://the-internet.herokuapp.com/dropdown",
    "https://the-internet.herokuapp.com/tables",
    "https://the-internet.herokuapp.com/hovers",
    "https://demoqa.com/buttons",
    "https://demoqa.com/radio-button",
    "https://demoqa.com/text-box",
    "https://practice.expandtesting.com/inputs",
]
DB_PATH = "../data/bench/bench_{ts}.sqlite"    # {ts} → a timestamp
TARGETS_PER_PAGE = 10
INSTRUCTIONS_PER_TARGET = 2
SEED = 42

DB_FILE = Path(DB_PATH.replace("{ts}", datetime.now().strftime("%Y%m%d_%H%M%S"))).resolve()
print(f"{len(URLS)} pages → {DB_FILE}")

11 pages → /home/rayounouna/Desktop/Thunder-X/data/bench/bench_20261101_231635.sqlite


## Step 1 · Read the page

Thunder-X's own `page_extractor` (`read_page`): every element with its tag, Chrome role, text, label, section headings, visibility and `backend_node_id` (Chrome's id for it, used in step 3 to find it again).

In [2]:
import asyncio
import json
import random
import re
import sqlite3
import sys
from collections import Counter

import pandas as pd

os.environ.setdefault("BROWSER_USE_LOGGING_LEVEL", "warning")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "false")
sys.path.insert(0, str(Path("..").resolve()))                  # Thunder-X's code
from browser_use import BrowserSession
from thunderx.page_extractor.services import read_page

pd.set_option("display.max_colwidth", 90)

## Step 2 · Pick targets

Each element gets the actions its role allows (a link → click, a text field → type, a checkbox → check, a heading → read…). Actions are taken in turn and a random element is picked for each, so targets are spread across actions. Hidden elements, elements in iframes, elements without a name (no text, no label) and destructive ones (delete, logout, buy…) are skipped.

**Category** (set after step 3): `duplicate_text` (another element that could take the same action has the same name) · `label_assoc` (a field named by its label) · `icon_only` (no visible text, not even in its children: only a label) · `unique`.

In [4]:
CLICKABLE = {"button", "link", "tab", "menuitem"}
TYPEABLE = {"textbox", "searchbox", "spinbutton"}
READ_TAGS = {"h1", "h2", "h3", "h4", "h5", "h6", "p", "span", "td", "th", "li", "label", "strong"}
DESTRUCTIVE = re.compile(r"\b(delete|remove|logout|log out|sign out|buy|pay|checkout|send)\b", re.I)
FAKE_TEXT = {"email": "jane.doe@example.com", "password": "Sunny-Day-42", "number": "42", "tel": "+216 20 123 456"}


def name_of(e):
    return e.text or e.label or ""


def actions_for(e):
    if not e.visible or e.frame_id or not name_of(e):
        return []
    if e.tag == "select":
        return ["select"]
    if e.role in ("checkbox", "radio", "switch"):
        return ["check"]
    if e.role in TYPEABLE or e.tag == "textarea":
        return ["type"]
    if e.role in CLICKABLE or e.tag in ("a", "button"):
        return ["click", "double_click", "right_click", "hover"]
    if e.text and e.tag in READ_TAGS and not e.interactive:
        return ["read"]
    return []


def category_of(t, has_text):
    """has_text: the element or its children show text (known after step 3)."""
    e = t["element"]
    if t["duplicate"]:
        return "duplicate_text"
    if e.tag in ("input", "select", "textarea") and e.label:
        return "label_assoc"
    if not has_text:
        return "icon_only"
    return "unique"


def pick_targets(url, elements, rng):
    pools = {}
    for e in elements:
        for action in actions_for(e):
            if action == "read" or not DESTRUCTIVE.search(name_of(e)):
                pools.setdefault(action, []).append(e)

    # duplicate = another element that could take the same action has the same name
    # (a <label> "Username" doesn't make the "Username" field a duplicate: it can't be typed into)
    name_counts = {action: Counter(name_of(e).lower() for e in pool) for action, pool in pools.items()}

    targets, used = [], set()
    while len(targets) < TARGETS_PER_PAGE and any(pools.values()):
        for action in pools:
            pools[action] = [e for e in pools[action] if e.index not in used]
            if not pools[action] or len(targets) == TARGETS_PER_PAGE:
                continue
            e = rng.choice(pools[action])
            used.add(e.index)
            value = FAKE_TEXT.get(e.attributes.get("type"), rng.choice(["Jane Doe", "hello world", "Tunis"])) if action == "type" else None
            targets.append({"url": url, "element": e, "action": action, "value": value,
                            "duplicate": name_counts[action][name_of(e).lower()] > 1})
    return targets

## Step 3 · XPath

Runs on the element itself (found by its `backend_node_id`). First it must be **actionable**: enabled, and once scrolled into view, the point at its centre is the element itself (no overlay). Then the rules, in order; the first XPath that matches **exactly one node, and that node is the target**, wins:

1. stable `id` · 2. `data-testid` · 3. `name` · 4. `aria-label` · 5. `<label for>` · 6. text inside its section heading · 7. text · 8. position (last resort)

Ids and names with a digit look generated (`react-select-3`, `ember123`): skipped.

In [5]:
XPATH_JS = r"""function (action) {
  const el = this;                                          // the element step 3 runs on
  const norm = s => (s || '').replace(/\s+/g, ' ').trim();
  const lit = s => s.includes("'") ? `"${s}"` : `'${s}'`;
  const stable = v => v && !/\d/.test(v);

  if (el.disabled) return {xpath: null, tried: ['disabled']};
  if (action !== 'read') {                                   // actionable: nothing on top of it
    el.scrollIntoView({block: 'center'});
    const r = el.getBoundingClientRect();
    const hit = document.elementFromPoint(r.x + r.width / 2, r.y + r.height / 2);
    const ok = hit && (hit === el || el.contains(hit) || (hit.closest('label') && hit.closest('label').control === el));
    if (!ok) return {xpath: null, tried: [`covered by <${hit ? hit.localName : 'nothing'}>`]};
  }

  const tag = el.localName, text = norm(el.textContent);
  const label = el.labels && el.labels[0];
  let heading = null;
  for (let a = el.parentElement; a && !heading; a = a.parentElement) {
    const h = a.querySelector('h1,h2,h3,h4,h5,h6,legend');
    if (h && !h.contains(el) && (h.compareDocumentPosition(el) & 4)) heading = h;
  }
  let position = '';
  for (let n = el; n !== document; n = n.parentNode) {
    const same = [...n.parentNode.children].filter(s => s.localName === n.localName);
    position = `/${n.localName}[${same.indexOf(n) + 1}]` + position;
  }

  const rules = [
    ['id',         stable(el.id) && `//*[@id=${lit(el.id)}]`],
    ['test-id',    el.dataset.testid && `//${tag}[@data-testid=${lit(el.dataset.testid)}]`],
    ['name',       stable(el.getAttribute('name')) && `//${tag}[@name=${lit(el.getAttribute('name'))}]`],
    ['aria-label', el.getAttribute('aria-label') && `//${tag}[@aria-label=${lit(el.getAttribute('aria-label'))}]`],
    ['label',      label && el.id && label.htmlFor === el.id && `//${tag}[@id=//label[normalize-space()=${lit(norm(label.textContent))}]/@for]`],
    ['section',    heading && text && text.length < 80 &&
                   `//*[${heading.localName}[normalize-space()=${lit(norm(heading.textContent))}]]//${tag}[normalize-space()=${lit(text)}]`],
    ['text',       text && text.length < 80 && `//${tag}[normalize-space()=${lit(text)}]`],
    ['position',   position],
  ];

  const tried = [];
  for (const [rule, xpath] of rules) {
    if (!xpath) continue;
    let found;
    try { found = document.evaluate(xpath, document, null, 7, null); } catch (e) { tried.push(`${rule}: invalid`); continue; }
    if (found.snapshotLength !== 1) { tried.push(`${rule}: ${found.snapshotLength} matches`); continue; }
    if (found.snapshotItem(0) !== el) { tried.push(`${rule}: another element`); continue; }
    const options = el.localName === 'select' ? [...el.options].slice(1).map(o => norm(o.text)).filter(Boolean) : [];
    return {xpath, rule, tried, options,         // options: for "select", to choose a value
            has_text: !!norm(el.innerText || el.textContent)};
  }
  return {xpath: null, tried};
}"""


async def find_xpath(browser, element, action):
    """Run XPATH_JS on the element, found by its backend_node_id."""
    cdp = await browser.get_or_create_cdp_session(focus=False)
    node = await cdp.cdp_client.send.DOM.resolveNode(
        params={"backendNodeId": element.backend_node_id}, session_id=cdp.session_id)
    answer = await cdp.cdp_client.send.Runtime.callFunctionOn(
        params={"functionDeclaration": XPATH_JS, "objectId": node["object"]["objectId"],
                "arguments": [{"value": action}], "returnByValue": True},
        session_id=cdp.session_id)
    if "exceptionDetails" in answer:
        raise RuntimeError(answer["exceptionDetails"].get("exception", {}).get("description", "JS error"))
    return answer["result"]["value"]

## Step 4 · Instruction from a template

Each action has a few phrasings; we pick `INSTRUCTIONS_PER_TARGET` of them. When the element's name isn't unique on the page, the instruction says where it is: *"Click on 'Edit' under the 'Data Tables' section"*.

In [7]:
TEMPLATES = {
    "click":        ["Click on '{name}'", "Click the '{name}' {kind}", "Press '{name}'"],
    "double_click": ["Double-click on '{name}'", "Double click the '{name}' {kind}"],
    "right_click":  ["Right-click on '{name}'", "Right click the '{name}' {kind}"],
    "hover":        ["Hover over '{name}'", "Move the mouse over '{name}'"],
    "type":         ["Type '{value}' into the '{name}' field", "Enter '{value}' in '{name}'", "Fill '{name}' with '{value}'"],
    "check":        ["Check '{name}'", "Tick the '{name}' box", "Select the '{name}' option"],
    "select":       ["Select '{value}' in the '{name}' dropdown", "Choose '{value}' from '{name}'"],
    "read":         ["Find the text '{name}'", "Locate '{name}'"],
}


def kind_of(e):
    """What a person calls it: "link", "button", or None (then templates with {kind} aren't used)."""
    if e.tag == "a" or e.role == "link":
        return "link"
    if e.tag == "button" or e.role == "button":
        return "button"
    return None


def instructions_for(t, rng):
    e = t["element"]
    name, kind = name_of(e)[:60], kind_of(e)
    where = f" under the '{e.sections[-1]}' section" if t["category"] == "duplicate_text" and e.sections else ""
    templates = [p for p in TEMPLATES[t["action"]] if kind or "{kind}" not in p]
    phrasings = rng.sample(templates, min(INSTRUCTIONS_PER_TARGET, len(templates)))
    return [p.format(name=name, value=t["value"], kind=kind) + where for p in phrasings]

## Run

One browser, one page at a time: read → pick → XPath → instructions.

In [8]:
browser = BrowserSession(headless=True, keep_alive=True, highlight_elements=False)
await browser.start()

ROWS, DROPPED = [], []
for url in URLS:
    print(f"\n── {url}")
    rng = random.Random(f"{SEED}{url}")
    try:
        await browser.navigate_to(url)
        await asyncio.sleep(2)                                                  # let the page render
        elements = (await read_page(browser)).elements                         # step 1
    except Exception as error:
        DROPPED.append((url, "page", str(error)[:120]))
        print(f"   ✘ can't read the page: {error}")
        continue

    for t in pick_targets(url, elements, rng):                                  # step 2
        what = f"{t['action']} <{t['element'].tag}> {name_of(t['element'])[:30]!r}"
        try:
            found = await find_xpath(browser, t["element"], t["action"])        # step 3
        except Exception as error:
            found = {"xpath": None, "tried": [f"error: {str(error)[:80]}"]}
        if found["xpath"] is None:
            DROPPED.append((url, what, "; ".join(found["tried"])))
            print(f"   ✘ {what:50} {'; '.join(found['tried'])}")
            continue
        if t["action"] == "select":
            if not found["options"]:
                DROPPED.append((url, what, "no option to choose"))
                continue
            t["value"] = rng.choice(found["options"])
        t["category"] = category_of(t, found["has_text"])
        category = t["category"] + ("_positional" if found["rule"] == "position" else "")
        for instruction in instructions_for(t, rng):                           # step 4
            ROWS.append({"url": url, "instruction": instruction, "expected_xpath": found["xpath"],
                         "action": t["action"], "value": t["value"], "category": category})
            print(f"   ✔ {instruction:50} {found['xpath']}")

await browser.kill()
print(f"\n{len(ROWS)} rows · {len(DROPPED)} targets dropped")

INFO     [BrowserSession] Setting viewport to 1920x1200 with device scale factor 1.0 whereas original device scale factor was 1.0



── https://www.selenium.dev/selenium/web/web-form.html
   ✔ Find the text 'Web form'                           //h1[normalize-space()='Web form']
   ✔ Locate 'Web form'                                  //h1[normalize-space()='Web form']
   ✔ Fill 'Text input' with 'Jane Doe'                  //*[@id='my-text-id']
   ✔ Enter 'Jane Doe' in 'Text input'                   //*[@id='my-text-id']
   ✔ Click on 'Submit'                                  //button[normalize-space()='Submit']
   ✔ Press 'Submit'                                     //button[normalize-space()='Submit']
   ✔ Double-click on 'File input'                       //input[@name='my-file']
   ✔ Double click the 'File input' button               //input[@name='my-file']
   ✔ Right-click on 'Return to index'                   //a[normalize-space()='Return to index']
   ✔ Right click the 'Return to index' link             //a[normalize-space()='Return to index']
   ✔ Choose 'One' from 'Dropdown (select)'              //select

INFO     [BrowserSession] 📢 on_BrowserStopEvent - Calling reset() (force=True, keep_alive=True)
INFO     [BrowserSession] [SessionManager] Cleared all owned data (targets, sessions, mappings)
INFO     [BrowserSession] ✅ Browser session reset complete
INFO     [BrowserSession] ✅ Browser session reset complete


   ✔ Find the text 'Web inputs page for Automation Testing Practice' //h1[normalize-space()='Web inputs page for Automation Testing Practice']
   ✔ Locate 'Web inputs page for Automation Testing Practice' //h1[normalize-space()='Web inputs page for Automation Testing Practice']
   ✔ Click on 'API Testing'                             //a[normalize-space()='API Testing']
   ✔ Click the 'API Testing' link                       //a[normalize-space()='API Testing']
   ✔ Double-click on 'PMP Practice'                     //a[normalize-space()='PMP Practice']
   ✔ Double click the 'PMP Practice' link               //a[normalize-space()='PMP Practice']
   ✔ Right click the 'Tips' link                        //a[normalize-space()='Tips']
   ✔ Right-click on 'Tips'                              //a[normalize-space()='Tips']
   ✔ Hover over 'Test Cases'                            //a[normalize-space()='Test Cases']
   ✔ Move the mouse over 'Test Cases'                   //a[normalize-space()='Test

## Step 5 · Save to SQLite

One standalone `.sqlite` file, one table `samples`. Written to a temporary file, then renamed (a crash never leaves half a DB).

In [10]:
DB_FILE.parent.mkdir(parents=True, exist_ok=True)
tmp = DB_FILE.with_suffix(".tmp")
tmp.unlink(missing_ok=True)

db = sqlite3.connect(tmp)
db.execute("""CREATE TABLE samples (
    id INTEGER PRIMARY KEY,
    url TEXT NOT NULL,
    instruction TEXT NOT NULL,
    expected_xpath TEXT NOT NULL,
    action TEXT NOT NULL,
    value TEXT,
    category TEXT,
    created_at TEXT)""")
now = datetime.now().isoformat(timespec="seconds")
db.executemany("INSERT INTO samples (url, instruction, expected_xpath, action, value, category, created_at) VALUES (?, ?, ?, ?, ?, ?, ?)",
               [(r["url"], r["instruction"], r["expected_xpath"], r["action"], r["value"], r["category"], now) for r in ROWS])
db.commit()
db.close()
os.replace(tmp, DB_FILE)

print(f"✔ {DB_FILE} · {DB_FILE.stat().st_size / 1024:.0f} KB · {len(ROWS)} rows")

✔ /home/rayounouna/Desktop/Thunder-X/data/bench/bench_20261101_231635.sqlite · 40 KB · 178 rows


## Summary

In [11]:
rows = pd.DataFrame(ROWS)
print("rows per action:  ", dict(rows["action"].value_counts()))
print("rows per category:", dict(rows["category"].value_counts()))
display(pd.DataFrame(DROPPED, columns=["url", "target", "why"]))
display(rows.sample(min(10, len(rows)), random_state=SEED)[["instruction", "expected_xpath", "category", "url"]])

rows per action:   {'read': np.int64(36), 'double_click': np.int64(32), 'right_click': np.int64(32), 'click': np.int64(30), 'hover': np.int64(20), 'type': np.int64(14), 'check': np.int64(12), 'select': np.int64(2)}
rows per category: {'unique': np.int64(122), 'duplicate_text_positional': np.int64(26), 'label_assoc': np.int64(22), 'label_assoc_positional': np.int64(6), 'unique_positional': np.int64(2)}


,url,target,why
0,https://www.selenium.dev/selenium/web/web-form.html,type <input> 'Disabled input',disabled
1,https://testpages.eviltester.com/styled/basic-html-form-test.html,hover <a> 'Contact Us',covered by <nothing>
2,https://the-internet.herokuapp.com/login,double_click <a> 'Fork me on GitHub',covered by <body>
3,https://the-internet.herokuapp.com/checkboxes,right_click <a> 'Fork me on GitHub',covered by <body>
4,https://the-internet.herokuapp.com/dropdown,click <a> 'Fork me on GitHub',covered by <body>
5,https://the-internet.herokuapp.com/hovers,click <a> 'Fork me on GitHub',covered by <body>


,instruction,expected_xpath,category,url
19,Click on 'Tool 3',/html[1]/body[1]/div[1]/div[1]/div[1]/aside[2]/div[4]/ul[1]/li[9]/a[1],duplicate_text_positional,https://testpages.eviltester.com/styled/basic-html-form-test.html
45,Fill 'Username' with 'hello world',//*[@id='username'],label_assoc,https://the-internet.herokuapp.com/login
140,Double-click on 'Web Tables',//a[normalize-space()='Web Tables'],unique,https://demoqa.com/text-box
30,Press 'Cookies 3',//a[normalize-space()='Cookies3'],unique,https://testpages.eviltester.com/styled/basic-html-form-test.html
67,Right click the 'Sauce Labs' link,//a[normalize-space()='Sauce Labs'],unique,https://the-internet.herokuapp.com/dropdown
16,Select the 'Default radio' option,/html[1]/body[1]/main[1]/div[1]/form[1]/div[1]/div[2]/div[3]/label[1]/input[1],label_assoc_positional,https://www.selenium.dev/selenium/web/web-form.html
119,Click on 'Web Tables',//a[normalize-space()='Web Tables'],unique,https://demoqa.com/radio-button
174,Double click the 'Demos' link,//*[@id='examples-dropdown'],unique,https://practice.expandtesting.com/inputs
109,Press 'Buttons',//a[normalize-space()='Buttons'],unique,https://demoqa.com/buttons
141,Double click the 'Web Tables' link,//a[normalize-space()='Web Tables'],unique,https://demoqa.com/text-box
